# 第21章　眼科② ― OCTと層構造ガイド型AI

**『医療診断支援AI開発　実装編 ― 本格実装（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 21.5　マルチチャネル入力の実装

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class LayerGuidedClassifier(nn.Module):
    def __init__(self, num_classes, in_channels=16):
        super().__init__()
        # EfficientNet-B3を出発点にする
        base = models.efficientnet_b3(weights="IMAGENET1K_V1")
        # 最初の畳み込みを16チャネル入力に置き換える。
        # ここで新しいConvをまっさらに作ると、ImageNet事前学習の重みを捨ててしまう。
        # RGB3chぶんは元の重みをそのまま引き継ぎ、増やした13chはその平均で初期化する。
        old_conv = base.features[0][0]
        new_conv = nn.Conv2d(
            in_channels, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            w = old_conv.weight                       # (out, 3, k, k)
            new_conv.weight[:, :3] = w                # RGBぶんは事前学習の重みを継承
            new_conv.weight[:, 3:] = w.mean(dim=1, keepdim=True)   # 残りは平均で初期化
        base.features[0][0] = new_conv
        self.backbone = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(1536, num_classes)

    def forward(self, x):   # x: (B, 16, H, W)
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        return self.head(x)

# 入力の組み立て：OCT画像(3ch) + 層構造マップ(13ch) = 16ch
def build_input(oct_rgb, layer_maps):
    return torch.cat([oct_rgb, layer_maps], dim=1)  # チャネル方向に結合

## 層厚マップを描く ― ETDRSグリッドと9セクター

In [ ]:
import numpy as np
def thickness_map(ilm_z, rpe_z, z_res_um):
    # z_res_um は軸方向の画素間隔[µm/画素]。装置の光学的な分解能ではない
    return (rpe_z - ilm_z) * z_res_um            # (H, W) の網膜厚マップ[µm]

def etdrs_sectors(tmap, fovea_xy, mm_per_px_xy, x_positive_is):
    """mm_per_px_xy: (x方向, y方向) の1画素あたりmm。黄斑キューブはAスキャン方向と
    Bスキャン方向で画素間隔が大きく違う（例：6×6mmを512×128で撮る）ため、
    等方と決め打ちすると外輪の上下セクターが空になり、平均がnanになる。
    x_positive_is: 画像のx正方向が "temporal"(耳側) か "nasal"(鼻側) か。
    右眼(OD)と左眼(OS)で反転するので、自分のデータで必ず確認して渡すこと。"""
    H, W = tmap.shape
    mm_x, mm_y = mm_per_px_xy
    yy, xx = np.mgrid[0:H, 0:W]
    dx = (xx - fovea_xy[0]) * mm_x                # 先に物理mmへ直してから
    dy = (yy - fovea_xy[1]) * mm_y                # 半径と角度を計算する
    r = np.hypot(dx, dy)
    ang = np.arctan2(dy, dx)
    rings = {"central": r <= 0.5, "inner": (r > 0.5) & (r <= 1.5),
             "outer": (r > 1.5) & (r <= 3.0)}
    # 耳側(T)/鼻側(N)は左右眼で反転する。x正方向がどちらを向くかは、
    # 左右眼と機器の書き出し規約で決まるため、呼び出し側が明示的に渡す。
    horiz_pos = "T" if x_positive_is == "temporal" else "N"   # x正方向のセクター
    horiz_neg = "N" if x_positive_is == "temporal" else "T"   # x負方向のセクター
    # 境界を半開区間にそろえ、各画素を必ず一つの象限だけへ割り当てる。
    quad = {
        "S": (ang >= -3*np.pi/4) & (ang < -np.pi/4),
        "I": (ang >= np.pi/4) & (ang < 3*np.pi/4),
        horiz_pos: (ang >= -np.pi/4) & (ang < np.pi/4),
        horiz_neg: (ang >= 3*np.pi/4) | (ang < -3*np.pi/4),
    }
    out = {"central": tmap[rings["central"]].mean()}
    for r_name in ("inner", "outer"):
        for q_name, q in quad.items():
            out[f"{r_name}_{q_name}"] = tmap[rings[r_name] & q].mean()
    return out                                   # 9セクターの平均厚